# Supplementary Classroom Tutorial: Parsing Raw AMPT Output & Particle Rapidity Analysis
=========================================================================================

In this classroom laboratory, we will learn **step-by-step** how to write a raw text file parser in Python to read the output files generated by the **A Multi-Phase Transport (AMPT)** model. We will write our own parser from scratch to inspect the data structure, extract particle momenta, and calculate and compare rapidity ($y$) distributions for different particle species.

### Pedagogical Split:
- **Instructor Demonstration:** Step-by-step walkthrough using the **7.7 GeV** dataset (`../Data/subsets/ampt_7.7_sub100.dat`).
- **Student Hands-on Task:** Replicate the parser and perform the analysis using the **39 GeV** dataset (`../Data/subsets/ampt_39_sub100.dat`), and make comparative physical observations.

---

## The AMPT Output Format (`ampt.dat`)
An AMPT data file is a text file containing consecutive events. Each event starts with an **Event Header** line, followed by $N$ **Particle Data** lines (where $N$ is the number of particles in that event).

### 1. Event Header Line Structure (Key Columns):
- Column 1: `Event Number`
- Column 3: `Number of Particles` (we use this to know how many subsequent lines to read as particles!)
- Column 4: `Impact Parameter` ($b$ in fm)
- Columns 5-6: `Number of Participants` ($N_{\mathrm{part1}}, N_{\mathrm{part2}}$)

### 2. Particle Data Line Structure (15 Columns, Key Columns):
- Column 1: `PID` (Particle ID according to PDG conventions: $211 = \pi^+$, $-211 = \pi^-$, $321 = K^+$, $2212 = p$, $111 = \pi^0$, $311 = K^0$, etc.)
- Column 2: `px` (x-momentum in GeV/c)
- Column 3: `py` (y-momentum in GeV/c)
- Column 4: `pz` (z-momentum in GeV/c)
- Column 5: `mass` (mass in GeV/$c^2$)
- Columns 6-9: Spacetime production coordinates ($x, y, z, t$ in fm, fm/c)

## Part 1: Instructor Demonstration — Parsing the 7.7 GeV Dataset

We will now write a simple, clean file parser in Python using standard file operations (`open()`, `.readline()`, and `.split()`) to read `../Data/subsets/ampt_7.7_sub100.dat`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Let's define a charge lookup mapping for common particles
CHARGE_MAP = {
    211: 1,     # pi+
    -211: -1,   # pi-
    111: 0,     # pi0
    321: 1,     # K+
    -321: -1,   # K-
    311: 0,     # K0
    -311: 0,    # anti-K0
    310: 0,     # K_Short
    130: 0,     # K_Long
    2212: 1,    # proton
    -2212: -1,  # antiproton
    2112: 0,    # neutron
    -2112: 0,   # antineutron
    11: -1,     # electron
    -11: 1,     # positron
    22: 0       # photon
}

def parse_ampt_file(filepath, max_events=100):
    """
    Parses the AMPT text file and extracts the event structures.
    """
    events = []
    with open(filepath, 'r') as f:
        for ev_idx in range(max_events):
            header_line = f.readline()
            if not header_line:
                break
            parts = header_line.split()
            if len(parts) < 4:
                break
            
            # Parse event header details
            event_num = int(parts[0])
            num_particles = int(parts[2])
            impact_param = float(parts[3])
            
            particles = []
            for _ in range(num_particles):
                p_line = f.readline()
                p_parts = p_line.split()
                
                pid = int(p_parts[0])
                px = float(p_parts[1])
                py = float(p_parts[2])
                pz = float(p_parts[3])
                mass = float(p_parts[4])
                
                particles.append({
                    'pid': pid,
                    'px': px,
                    'py': py,
                    'pz': pz,
                    'mass': mass
                })
                
            events.append({
                'event_num': event_num,
                'impact_param': impact_param,
                'particles': particles
            })
    return events

# Load the 7.7 GeV dataset
file_7_7 = "../Data/subsets/ampt_7.7_sub100.dat"
events_7_7 = parse_ampt_file(file_7_7, max_events=100)
print(f"Successfully parsed {len(events_7_7)} events from 7.7 GeV dataset.")
print(f"Example event 1 has {len(events_7_7[0]['particles'])} particles, b = {events_7_7[0]['impact_param']:.2f} fm.")

### Calculating Kinematics and Species Classification
Let's write functions to compute transverse momentum $p_T$ and rapidity $y$:
$$p_T = \sqrt{p_x^2 + p_y^2}$$
$$y = \frac{1}{2} \ln\left( \frac{E + p_z}{E - p_z} \right), \quad \text{where } E = \sqrt{p_T^2 + p_z^2 + m^2}$$

In [ ]:
def calculate_pt(px, py):
    return np.sqrt(px**2 + py**2)

def calculate_rapidity(px, py, pz, mass):
    pt = calculate_pt(px, py)
    E = np.sqrt(pt**2 + pz**2 + mass**2)
    # Protect against divide-by-zero or negative arguments inside log
    numerator = E + pz
    denominator = np.maximum(E - pz, 1e-15)
    return 0.5 * np.log(numerator / denominator)

# Let's filter and analyze species at 7.7 GeV
y_pions = []
y_kaons = []
y_protons = []
y_charged = []
y_neutral = []

for ev in events_7_7:
    for p in ev['particles']:
        y = calculate_rapidity(p['px'], p['py'], p['pz'], p['mass'])
        pid_abs = np.abs(p['pid'])
        
        # Species classification
        if pid_abs == 211: # Charged Pions
            y_pions.append(y)
        elif pid_abs == 321: # Charged Kaons
            y_kaons.append(y)
        elif pid_abs == 2212: # Protons
            y_protons.append(y)
            
        # Charged vs. Neutral classification
        charge = CHARGE_MAP.get(p['pid'], None)
        if charge is not None:
            if charge != 0:
                y_charged.append(y)
            else:
                y_neutral.append(y)

# Create demonstration plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Plot 1: Identified Charged Particles
bins = np.linspace(-3.0, 3.0, 30)
ax1.hist(y_pions, bins=bins, histtype='step', color='blue', linewidth=1.8, label=r'Pions ($\pi^\pm$)')
ax1.hist(y_kaons, bins=bins, histtype='step', color='green', linewidth=1.8, label=r'Kaons ($K^\pm$)')
ax1.hist(y_protons, bins=bins, histtype='step', color='red', linewidth=1.8, label=r'Protons ($p/\bar{p}$)')
ax1.set_xlabel('Rapidity y', fontsize=12)
ax1.set_ylabel('dI/dy', fontsize=12)
ax1.set_title('Identified Charged Particle Rapidity (7.7 GeV)', fontsize=13)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(frameon=True)

# Plot 2: Charged vs. Neutral
ax2.hist(y_charged, bins=bins, histtype='stepfilled', color='#6366f1', alpha=0.4, edgecolor='#4f46e5', linewidth=1.8, label='All Charged Particles')
ax2.hist(y_neutral, bins=bins, histtype='step', color='orange', linewidth=1.8, label='All Neutral Particles')
ax2.set_xlabel('Rapidity y', fontsize=12)
ax2.set_ylabel('dI/dy', fontsize=12)
ax2.set_title('Charged vs. Neutral Multiplicities (7.7 GeV)', fontsize=13)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(frameon=True)

plt.show()

## Part 2: Student Hands-on Lab — Parsing the 39 GeV Dataset

Now it is your turn! You will adapt the parser and run the analysis on the higher energy **39 GeV** dataset. This will allow you to see how particle yields and longitudinal shapes change due to the collision energy.

### Your Tasks:
1. Parse the 39 GeV dataset `../Data/subsets/ampt_39_sub100.dat` for 100 events.
2. Extract $y$ distributions for charged pions, charged kaons, and protons.
3. Extract $y$ distributions for all charged particles vs. neutral particles.
4. Plot the results in a 2-panel chart (just like the demonstration above).
5. **Physical Discussion Questions:**
   - Compare the peak value of $dN/dy$ for pions between 7.7 GeV and 39 GeV. What does this tell you about particle production scaling?
   - Look at the width of the rapidity distributions. How does the width of the distribution scale with beam energy, and how does this relate to the kinematic beam rapidity limit $y_{\mathrm{beam}}$?

In [ ]:
# TODO: Define path to the 39 GeV file
file_39 = "../Data/subsets/ampt_39_sub100.dat"

# TODO: Parse the 39 GeV dataset for 100 events


# TODO: Initialize list arrays for pions, kaons, protons, charged, and neutral particles


# TODO: Iterate over events, compute rapidity, and classify particles


# TODO: Plot the identified particles and charged vs. neutral distributions for 39 GeV
# Hint: Use the 7.7 GeV plotting code as a reference template!


### Student Physical Discussion Responses
*(Write your brief explanations and comparisons here)*